# Search-o1 with QwQ-32B-Preview-AWQ (4-bit)

## 0. Environment Setup

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil
DRIVE_DIR = '/content/drive/MyDrive/search-o1-reproduction'
os.makedirs(f'{DRIVE_DIR}/outputs_32b', exist_ok=True)

print(f'Drive dir: {DRIVE_DIR}')
print(f'Contents: {os.listdir(DRIVE_DIR)}')

Mounted at /content/drive
Drive dir: /content/drive/MyDrive/search-o1-reproduction
Contents: ['outputs', 'data', 'bing_search.py', 'bing_search_brave_replacement.py', 'outputs_gpqa', 'outputs_all', 'outputs_32b', 'api_config.json']


In [ ]:
# Clone repo
%cd /content
if not os.path.exists('/content/Search-o1'):
    !git clone https://github.com/RUC-NLPIR/Search-o1.git
%cd /content/Search-o1

/content
Cloning into 'Search-o1'...
remote: Enumerating objects: 391, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 391 (delta 168), reused 156 (delta 156), pack-reused 211 (from 1)
Receiving objects: 100% (391/391), 4.51 MiB | 12.58 MiB/s, done.
Resolving deltas: 100% (195/195), done.
/content/Search-o1


In [ ]:
# Install dependencies
!pip install -q torch==2.5.1 transformers==4.46.1 sentencepiece==0.2.0 tqdm nltk
!pip install -q vllm==0.6.4
!pip install -q datasets trafilatura requests autoawq

# Verify GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 144.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 194.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 142.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA RTX PRO 6000 Blackwell Server Edition with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA RTX PRO 6000 Blackwell Server Edition GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [ ]:
# Fix pyext (dummy module)
import sys, types
import pyext


## 1. Restore Brave Search Drop-in Replacement

In [ ]:
%%writefile /content/Search-o1/scripts/bing_search.py

"""
Drop-in replacement for bing_search.py
Uses Brave Search API + snippet-based content
"""

import time
import requests
import re
from typing import Optional, List, Dict

def _truncate_query(query, max_len=200):
    """Clean and truncate query for Brave Search API."""
    for pattern in [r'\s*Choices:\s*', r'\s*\(A\)', r'\s*\ba\)', r'\s*Options:', r'\s*A\.']:
        parts = re.split(pattern, query, maxsplit=1)
        query = parts[0]
    query = re.sub(r'\s+', ' ', query).strip()
    if len(query) > max_len:
        sentences = re.split(r'[.?!]\s+', query)
        query = sentences[0]
        if len(query) > max_len:
            query = query[:max_len]
    return query

def bing_web_search(query, subscription_key, endpoint, market='en-US', language='en', timeout=20):
    query = _truncate_query(query)
    headers = {
        'Accept': 'application/json',
        'Accept-Encoding': 'gzip',
        'X-Subscription-Token': subscription_key
    }
    params = {
        'q': query,
        'count': 10,
        'search_lang': 'en',
        'country': 'us'
    }
    for attempt in range(3):
        try:
            resp = requests.get(
                'https://api.search.brave.com/res/v1/web/search',
                headers=headers, params=params, timeout=timeout
            )
            resp.raise_for_status()
            brave_data = resp.json()
            bing_compatible = {'webPages': {'value': []}}
            for r in brave_data.get('web', {}).get('results', []):
                bing_compatible['webPages']['value'].append({
                    'name': r.get('title', ''),
                    'url': r.get('url', ''),
                    'snippet': r.get('description', ''),
                })
            return bing_compatible
        except Exception as e:
            print(f'[Brave] Error attempt {attempt+1}: {e}')
            time.sleep(3 * (attempt + 1))
    return {'webPages': {'value': []}}


def extract_relevant_info(search_results):
    if not search_results or 'webPages' not in search_results:
        return []
    return [
        {'title': item.get('name',''), 'url': item.get('url',''), 'snippet': item.get('snippet','')}
        for item in search_results['webPages'].get('value', [])
    ]


def fetch_page_content(urls, max_workers=8, use_jina=False, jina_api_key=None, snippets: Optional[dict] = None):
    if isinstance(urls, str):
        urls = [urls]
    contents = {}
    for url in urls:
        if snippets and url in snippets:
            contents[url] = snippets[url]
    return contents


def extract_snippet_with_context(raw_context, snippet=None, context_chars=3000, max_results=10):
    """
    Compatible with BOTH calling patterns:
    1. run_naive_rag.py: extract_snippet_with_context(raw_context, snippet, context_chars=max_doc_len)
       - Returns (success: bool, context: str)
    2. run_search_o1.py: extract_snippet_with_context(search_results, max_results=10)
       - Returns dict of {url: snippet}
    """
    # Pattern 1: called from run_naive_rag.py with (raw_context, snippet, context_chars)
    if isinstance(raw_context, str):
        if snippet and snippet in raw_context:
            idx = raw_context.find(snippet)
            start = max(0, idx - context_chars // 2)
            end = min(len(raw_context), idx + len(snippet) + context_chars // 2)
            return True, raw_context[start:end]
        elif raw_context:
            return True, raw_context[:context_chars]
        elif snippet:
            return True, snippet
        else:
            return False, ''

    # Pattern 2: called from run_search_o1.py with (search_results, max_results)
    if isinstance(raw_context, dict):
        snippets = {}
        if 'webPages' not in raw_context:
            return snippets
        for item in raw_context['webPages'].get('value', [])[:max_results]:
            url = item.get('url', '')
            snip = item.get('snippet', '')
            if url and snip:
                snippets[url] = snip
        return snippets

    return False, ''

print('✅ bing_search.py (Brave replacement) written')

Overwriting /content/Search-o1/scripts/bing_search.py


## 2. Set Brave API Key

In [ ]:
# Try loading from Drive first, otherwise ask for input
config_path = f'{DRIVE_DIR}/api_config.json'
if os.path.exists(config_path):
    config = json.load(open(config_path))
    BRAVE_KEY = config['brave_api_key']
    print(f'✅ Brave API key loaded from Drive: {BRAVE_KEY[:8]}...')
else:
    BRAVE_KEY = input('Enter your Brave API key: ')
    json.dump({'brave_api_key': BRAVE_KEY}, open(config_path, 'w'))
    print('✅ Brave API key saved to Drive')

os.environ['BRAVE_KEY'] = BRAVE_KEY

## 3. Prepare GPQA Diamond Dataset

In [ ]:
!pip install -q numpy==1.26.4

In [ ]:
%cd /content/Search-o1/scripts

import json, random, os

os.makedirs('data/GPQA', exist_ok=True)

# Check if data already exists from previous runs
if os.path.exists('data/GPQA/diamond.json'):
    with open('data/GPQA/diamond.json') as f:
        existing = json.load(f)
    print(f'✅ GPQA already exists: {len(existing)} questions')
else:
    # Try loading from HuggingFace
    from datasets import load_dataset

    try:
        ds = load_dataset('Idavidrein/gpqa', 'gpqa_diamond', split='train',
                          trust_remote_code=True)
        print(f'✅ Loaded from HF: {len(ds)} questions')
    except Exception as e:
        print(f'⚠️ Gated dataset error: {e}')
        print('Run: !huggingface-cli login')
        print('Then re-run this cell')
        ds = None

    if ds is not None:
        filtered_data = []
        for idx, row in enumerate(ds):
            answers = [
                ('Correct Answer', row['Correct Answer']),
                ('Incorrect Answer 1', row['Incorrect Answer 1']),
                ('Incorrect Answer 2', row['Incorrect Answer 2']),
                ('Incorrect Answer 3', row['Incorrect Answer 3']),
            ]
            random.shuffle(answers)
            choices = ['A', 'B', 'C', 'D']
            formatted_answers = []
            correct_choice = None
            for i, (label, answer) in enumerate(answers):
                choice = choices[i]
                formatted_answers.append((choice, answer))
                if label == 'Correct Answer':
                    correct_choice = choice

            formatted_choices = '\n'.join([f'({c}) {a}' for c, a in formatted_answers])
            question_with_choices = f"{row['Question']} Choices:\n{formatted_choices}\n"

            filtered_data.append({
                'id': idx,
                'Question': question_with_choices,
                'Subdomain': row.get('Subdomain', ''),
                'High-level domain': row.get('High-level domain', ''),
                'Correct Answer': row['Correct Answer'],
                'Incorrect Answer 1': row['Incorrect Answer 1'],
                'Incorrect Answer 2': row['Incorrect Answer 2'],
                'Incorrect Answer 3': row['Incorrect Answer 3'],
                'Correct Choice': correct_choice,
            })

        with open('data/GPQA/diamond.json', 'w', encoding='utf-8') as f:
            json.dump(filtered_data, f, indent=4, ensure_ascii=False)
        print(f'✅ Saved {len(filtered_data)} questions to data/GPQA/diamond.json')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Idavidrein/gpqa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Idavidrein/gpqa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


/content/Search-o1/scripts


gpqa_diamond.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/198 [00:00<?, ? examples/s]

✅ Loaded from HF: 198 questions
✅ Saved 198 questions to data/GPQA/diamond.json


In [ ]:
!huggingface-cli login --token YOUR_HF_TOKEN

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `colab2` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `colab2`


## 4. Quick Model Test (3 questions)

First verify QwQ-32B-Preview-AWQ loads and generates properly.

In [ ]:
# Quick test: Direct Reasoning with 3 questions to verify model works
%cd /content/Search-o1/scripts

MODEL_32B = 'KirillR/QwQ-32B-Preview-AWQ'

!python run_direct_gen.py \
    --dataset_name gpqa \
    --split diamond \
    --subset_num 3 \
    --model_path {MODEL_32B} \
    --temperature 0.7 \
    --top_p 0.8 \
    --top_k 20 \
    --repetition_penalty 1.05 \
    --max_tokens 16384

In [ ]:
# Check test output
import glob, json

test_files = sorted(glob.glob('outputs/**/diamond*.json', recursive=True))
print('Output files:', test_files)

if test_files:
    # Find the most recent 3-question test
    for f in test_files:
        with open(f) as fh:
            data = json.load(fh)
        if len(data) == 3:
            print(f'\n--- {f} (3-question test) ---')
            for item in data:
                output = item.get('Output', '')
                print(f"Q: {item['Question'][:80]}...")
                print(f"Output (first 200 chars): {output[:200]}")
                print(f"Output (last 200 chars): {output[-200:]}")
                print()

## 5. Run Search-o1 with QwQ-32B-Preview-AWQ on GPQA

This is the key experiment. If the 32B model triggers `<|begin_search_query|>` tokens, it proves the 7B model's failure was due to model capability.

In [ ]:
!python -c "import site; print(site.getsitepackages())"

['/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/lib/python3.12/dist-packages']


In [ ]:
import subprocess, sys

result = subprocess.run([sys.executable, '-c', 'import site; print(site.getsitepackages()[0])'],
                       capture_output=True, text=True)
site_path = result.stdout.strip()
print(f'Site packages: {site_path}')

!mkdir -p {site_path}/pyext
!echo "class RuntimeModule: pass" > {site_path}/pyext/__init__.py
!python -c "from pyext import RuntimeModule; print('✅ pyext patched')"

Site packages: /usr/local/lib/python3.12/dist-packages
✅ pyext patched


In [ ]:
!nvidia-smi

Sat Mar  7 23:37:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip uninstall vllm -y
!pip install vllm --upgrade
!pip install torch --upgrade
!python -c "import vllm; print(f'vLLM version: {vllm.__version__}')"
!python -c "import torch; print(f'PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}')"

Found existing installation: vllm 0.6.4
Uninstalling vllm-0.6.4:
  Successfully uninstalled vllm-0.6.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.9/432.9 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 165.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 117.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 146.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 830.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/usr/local/lib/python3.12/dist-packages/vllm/env_override.py", line 87, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 2247, in <module>
    from torch import (
  File "/usr/local/lib/python3.12/dist-packages/torch/nested/__init__.py", line 21, in <module>
    from ._internal.nested_tensor import _rebuild_njt, NestedTensor as _NestedTensor
  File "/usr/local/lib/python3.12/dist-packages/torch/nested/_internal/nested_tensor.py", line 7, in <module>
    from torch.nested._internal.nested_int import NestedIntNode
  File "/usr/local

In [ ]:
%cd /content/Search-o1/scripts

!python run_search_o1.py \
    --dataset_name gpqa \
    --split diamond \
    --model_path Qwen/QwQ-32B-Preview \
    --bing_subscription_key {BRAVE_KEY} \
    --max_search_limit 5 \
    --max_turn 10 \
    --top_k 10 \
    --max_doc_len 3000 \
    --temperature 0.7 \
    --top_p 0.8 \
    --repetition_penalty 1.05 \
    --max_tokens 16384

Streaming output truncated to the last 5000 lines.
model-00016-of-00017.safetensors:   0% 0.00/3.90G [01:38<?, ?B/s]
model-00016-of-00017.safetensors:   0% 0.00/3.90G [01:38<?, ?B/s]
model-00016-of-00017.safetensors:   0% 0.00/3.90G [01:38<?, ?B/s]
model-00016-of-00017.safetensors:   0% 0.00/3.90G [01:38<?, ?B/s]
model-00016-of-00017.safetensors:   0% 0.00/3.90G [01:38<?, ?B/s]
model-00016-of-00017.safetensors:   0% 0.00/3.90G [01:38<?, ?B/s]
model-00016-of-00017.safetensors:   0% 0.00/3.90G [01:38<?, ?B/s]
model-00016-of-00017.safetensors:   0% 0.00/3.90G [01:38<?, ?B/s]
model-00016-of-00017.safetensors:   0% 0.00/3.90G [01:38<?, ?B/s]
model-00016-of-00017.safetensors:   0% 0.00/3.90G [01:38<?, ?B/s]
model-00016-of-00017.safetensors:   0% 0.00/3.90G [01:38<?, ?B/s]
model-00016-of-00017.safetensors:   0% 0.00/3.90G [01:38<?, ?B/s]
model-00016-of-00017.safetensors:   0% 0.00/3.90G [01:38<?, ?B/s]
model-00016-of-00017.safetensors:   0% 0.00/3.90G [01:38<?, ?B/s]
model-00016-of-00017.safe

In [ ]:
%cd /content/Search-o1/scripts

!python run_search_o1.py \
    --dataset_name gpqa \
    --split diamond \
    --model_path Qwen/QwQ-32B-Preview \
    --bing_subscription_key YOUR_BRAVE_API_KEY \
    --max_search_limit 5 \
    --max_turn 10 \
    --top_k 10 \
    --max_doc_len 3000 \
    --temperature 0.7 \
    --top_p 0.8 \
    --repetition_penalty 1.05 \
    --max_tokens 16384

Streaming output truncated to the last 5000 lines.
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
model-00017-of-00017.safetensors:  17% 536M/3.10G [00:08<00:18, 140MB/s]
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
model-00017-of-00017.safetensors:  17% 536M/3.10G [00:08<00:18, 140MB/s]
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
model-00017-of-00017.safetensors:  17% 536M/3.10G [00:08<00:18, 140MB/s]
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
model-00017-of-00017.safetensors:  17% 536M/3.10G [00:08<00:18, 140MB/s]
(EngineCore_DP0 pid=18787) 
(EngineCore_DP0 pid=18787) 
(EngineCore_D

In [ ]:
import subprocess, sys

result = subprocess.run([sys.executable, '-c', 'import site; print(site.getsitepackages()[0])'],
                       capture_output=True, text=True)
site_path = result.stdout.strip()

!mkdir -p {site_path}/pyext
!echo "class RuntimeModule: pass" > {site_path}/pyext/__init__.py
!python -c "from pyext import RuntimeModule; print('✅ pyext patched')"

✅ pyext patched


In [ ]:
!pip uninstall vllm -y
!pip install vllm --upgrade
!pip install torch --upgrade
!python -c "import vllm; print(f'vLLM: {vllm.__version__}')"
!python -c "import torch; print(f'PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}')"

Found existing installation: vllm 0.6.4
Uninstalling vllm-0.6.4:
  Successfully uninstalled vllm-0.6.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.9/432.9 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 187.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 190.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 142.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 834.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

Traceback (most recent call last):
  File "/usr/lib/python3.12/importlib/metadata/__init__.py", line 819, in read_text
^C


In [ ]:
%cd /content/Search-o1/scripts

# ============================================
# Naive RAG - QwQ-32B-Preview
# ============================================
print("=" * 60)
print("GPQA - Naive RAG (QwQ-32B)")
print("=" * 60)

!python run_naive_rag.py \
    --dataset_name gpqa \
    --split diamond \
    --model_path Qwen/QwQ-32B-Preview \
    --bing_subscription_key YOUR_BRAVE_API_KEY \
    --top_k 10 \
    --max_doc_len 3000 \
    --temperature 0.7 \
    --top_p 0.8 \
    --repetition_penalty 1.05 \
    --max_tokens 16384

# ============================================
# RAgent - QwQ-32B-Preview
# ============================================
print("\n" + "=" * 60)
print("GPQA - RAgent (QwQ-32B)")
print("=" * 60)

!python run_rag_agent.py \
    --dataset_name gpqa \
    --split diamond \
    --model_path Qwen/QwQ-32B-Preview \
    --bing_subscription_key YOUR_BRAVE_API_KEY \
    --max_search_limit 5 \
    --max_url_fetch 5 \
    --max_turn 10 \
    --top_k 10 \
    --temperature 0.7 \
    --top_p 0.8 \
    --repetition_penalty 1.05 \
    --max_tokens 16384

# ============================================
# Save all results
# ============================================
!cp -r /content/Search-o1/scripts/outputs/ /content/drive/MyDrive/search-o1-reproduction/outputs_32b/

print("\n" + "=" * 60)
print("ALL 32B EXPERIMENTS COMPLETE")
print("=" * 60)

import glob, json
for f in sorted(glob.glob('outputs/**/*.metrics*.json', recursive=True)):
    with open(f) as fh:
        m = json.load(fh)
    print(f'\n{f}')
    print(f"  EM={m['overall']['em']:.3f}  Valid={m['overall']['num_valid_answer']}")

Streaming output truncated to the last 5000 lines.
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
model-00001-of-00017.safetensors:  39% 1.51G/3.92G [03:19<01:56, 20.6MB/s]
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
model-00001-of-00017.safetensors:  39% 1.51G/3.92G [03:19<01:56, 20.6MB/s]
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
model-00001-of-00017.safetensors:  39% 1.51G/3.92G [03:19<01:56, 20.6MB/s]
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
model-00001-of-00017.safetensors:  39% 1.51G/3.92G [03:19<01:56, 20.6MB/s]
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
(EngineCore_DP0 pid=8416) 
model-00001-of-00017.safetensors:  39% 1.51G/3.92G [03:

## 6. Analyze Results

In [ ]:
# Find the Search-o1 output file
import glob, json

search_files = sorted(glob.glob('outputs/**/*search_o1*/*.json', recursive=True))
print('Search-o1 output files:')
for f in search_files:
    if 'metrics' not in f and 'info_extract' not in f:
        print(f'  {f}')

Search-o1 output files:
  outputs/gpqa.qwq.search_o1/diamond.3.8,2:43.json


In [ ]:
# ============================================
# KEY ANALYSIS: Did the 32B model trigger search?
# ============================================

# Find the latest search_o1 output (non-metrics, non-info_extract)
search_output = None
for f in sorted(glob.glob('outputs/**/*search_o1*/*.json', recursive=True), reverse=True):
    if 'metrics' not in f and 'info_extract' not in f:
        search_output = f
        break

if search_output:
    with open(search_output) as fh:
        data = json.load(fh)

    search_count = 0
    total_searches = 0
    for item in data:
        output = item.get('Output', '')
        n = output.count('<|begin_search_query|>')
        if n > 0:
            search_count += 1
            total_searches += n

    print(f'Questions that triggered search: {search_count} / {len(data)}')
    print(f'Total search queries: {total_searches}')
    print(f'Avg searches per question: {total_searches / len(data):.2f}')

    # Show examples
    for i in range(min(3, len(data))):
        output = data[i].get('Output', '')
        has_search = '<|begin_search_query|>' in output
        print(f'\n{"="*60}')
        print(f'Q{i}: search={has_search}')
        print(f'Output start: {output[:300]}')
        print(f'Output end: {output[-300:]}')
else:
    print('❌ No search_o1 output found')

Questions that triggered search: 111 / 198
Total search queries: 333
Avg searches per question: 1.68

Q0: search=True
Output start: Alright, I've got this question about quantum states and their energy differences, and I need to figure out which energy difference would allow us to clearly distinguish between two energy levels given their lifetimes. The lifetimes are 10^-9 seconds and 10^-8 seconds, and the choices for energy dif
Output end: nd_search_result|>

Okay, according to this search result, option (a) \(10^{-4}\) eV is the only one that exceeds the sum of energy uncertainties, making it the correct choice for clear distinction.

Therefore, the answer should be \boxed{A}

**Final Answer**

\[ \boxed{a} \] <|end_search_result|>



Q1: search=True
Output start: Alright, I've got this problem here about some chemical reactions, and I need to figure out how many carbon atoms are in the final product, product 3. Let's break this down step by step.

First, there's trans-cinnamaldehyde

In [ ]:
# Show metrics
for f in sorted(glob.glob('outputs/**/*search_o1*/*.metrics*.json', recursive=True)):
    with open(f) as fh:
        metrics = json.load(fh)
    print(f'\n--- {f} ---')
    print(json.dumps(metrics, indent=2))


--- outputs/gpqa.qwq.search_o1/diamond.3.8,2:43.metrics.json ---
{
  "overall": {
    "em": 0.5555555555555556,
    "acc": 0.5707070707070707,
    "f1": 0.5555555555555556,
    "math_equal": 0.5555555555555556,
    "num_valid_answer": "179 of 198",
    "query_latency": "53258 ms"
  },
  "per_domain": {
    "Physics": {
      "em": 0.7325581395348837,
      "acc": 0.7558139534883721,
      "f1": 0.7325581395348837,
      "math_equal": 0.7325581395348837,
      "num_valid_answer": "75 of 86"
    },
    "Chemistry": {
      "em": 0.3978494623655914,
      "acc": 0.40860215053763443,
      "f1": 0.3978494623655914,
      "math_equal": 0.3978494623655914,
      "num_valid_answer": "86 of 93"
    },
    "Biology": {
      "em": 0.5263157894736842,
      "acc": 0.5263157894736842,
      "f1": 0.5263157894736842,
      "math_equal": 0.5263157894736842,
      "num_valid_answer": "18 of 19"
    }
  }
}


## 7. (Optional) Run Direct Reasoning with 32B for backoff baseline

In [ ]:
# Run Direct Reasoning with 32B (needed for proper backoff comparison)
%cd /content/Search-o1/scripts

MODEL_32B = 'KirillR/QwQ-32B-Preview-AWQ'

!python run_direct_gen.py \
    --dataset_name gpqa \
    --split diamond \
    --model_path {MODEL_32B} \
    --temperature 0.7 \
    --top_p 0.8 \
    --top_k 20 \
    --repetition_penalty 1.05 \
    --max_tokens 16384

In [ ]:
# Check direct reasoning results
for f in sorted(glob.glob('outputs/**/*.metrics.json', recursive=True)):
    with open(f) as fh:
        m = json.load(fh)
    print(f'{f}')
    print(f"  em={m['overall']['em']:.3f}  valid={m['overall']['num_valid_answer']}")
    print()

In [ ]:
%cd /content/Search-o1/scripts

# ============================================
# Naive RAG - QwQ-32B-Preview
# ============================================
print("=" * 60)
print("GPQA - Naive RAG (QwQ-32B)")
print("=" * 60)

!python run_naive_rag.py \
    --dataset_name gpqa \
    --split diamond \
    --model_path Qwen/QwQ-32B-Preview \
    --bing_subscription_key YOUR_BRAVE_API_KEY \
    --top_k 10 \
    --max_doc_len 3000 \
    --temperature 0.7 \
    --top_p 0.8 \
    --repetition_penalty 1.05 \
    --max_tokens 16384

# ============================================
# RAgent - QwQ-32B-Preview
# ============================================
print("\n" + "=" * 60)
print("GPQA - RAgent (QwQ-32B)")
print("=" * 60)

!python run_rag_agent.py \
    --dataset_name gpqa \
    --split diamond \
    --model_path Qwen/QwQ-32B-Preview \
    --bing_subscription_key YOUR_BRAVE_API_KEY \
    --max_search_limit 5 \
    --max_url_fetch 5 \
    --max_turn 10 \
    --top_k 10 \
    --temperature 0.7 \
    --top_p 0.8 \
    --repetition_penalty 1.05 \
    --max_tokens 16384

# ============================================
# Save all results
# ============================================
!cp -r /content/Search-o1/scripts/outputs/ /content/drive/MyDrive/search-o1-reproduction/outputs_32b/

print("\n" + "=" * 60)
print("ALL 32B EXPERIMENTS COMPLETE")
print("=" * 60)

import glob, json
for f in sorted(glob.glob('outputs/**/*.metrics*.json', recursive=True)):
    with open(f) as fh:
        m = json.load(fh)
    print(f'\n{f}')
    print(f"  EM={m['overall']['em']:.3f}  Valid={m['overall']['num_valid_answer']}")

## 8. Apply Backoff for Search-o1 32B (Optional)

In [ ]:
# Patch evaluate.py to use our 32B direct reasoning output for backoff
import glob

# Find the 32B direct output
direct_files = [f for f in glob.glob('outputs/**/*direct*/*.json', recursive=True)
                if 'metrics' not in f and 'QwQ' in f.lower() or 'qwq' in f.lower() or '32b' in f.lower()]

# If naming doesn't include model name, find the latest direct output with 198 items
if not direct_files:
    for f in sorted(glob.glob('outputs/**/*direct*/diamond*.json', recursive=True), reverse=True):
        if 'metrics' not in f:
            with open(f) as fh:
                data = json.load(fh)
            if len(data) == 198:
                direct_files = [f]
                break

print(f'Direct reasoning output candidates: {direct_files}')

if direct_files:
    direct_path = direct_files[0]
    print(f'\nUsing: {direct_path}')

    # Patch evaluate.py
    with open('evaluate.py', 'r') as f:
        content = f.read()

    # Replace ALL GPQA backoff paths with our 32B direct output
    import re
    # Find the GPQA section and replace the normal_output_path
    lines = content.split('\n')
    for i, line in enumerate(lines):
        if 'gpqa' in line.lower() and 'normal_output_path' in line and '.json' in line:
            old_path = line.strip()
            new_line = f"        normal_output_path = './{direct_path}'"
            lines[i] = new_line
            print(f'Replaced line {i}: {old_path} -> {new_line.strip()}')

    with open('evaluate.py', 'w') as f:
        f.write('\n'.join(lines))
    print('\n✅ evaluate.py patched')
else:
    print('⚠️ No 32B direct reasoning output found. Run Section 7 first.')

In [ ]:
# Apply backoff to Search-o1 32B results
search_o1_output = None
for f in sorted(glob.glob('outputs/**/*search_o1*/diamond*.json', recursive=True), reverse=True):
    if 'metrics' not in f and 'info_extract' not in f:
        search_o1_output = f
        break

if search_o1_output:
    print(f'Applying backoff to: {search_o1_output}')
    !python evaluate.py --output_path {search_o1_output} --apply_backoff
else:
    print('❌ No Search-o1 output found')

## 9. Final Comparison: 7B vs 32B

In [ ]:
# ============================================
# Complete results comparison
# ============================================
print('=' * 70)
print('ALL RESULTS')
print('=' * 70)

for f in sorted(glob.glob('outputs/**/*.metrics*.json', recursive=True)):
    with open(f) as fh:
        m = json.load(fh)
    overall = m['overall']
    print(f'{f}')
    print(f"  EM={overall['em']:.3f}  F1={overall.get('f1', 0):.3f}  Valid={overall['num_valid_answer']}")
    if 'per_domain' in m:
        for domain, dm in m['per_domain'].items():
            print(f"    {domain}: EM={dm['em']:.3f}")
    print()

print('\n' + '=' * 70)
print('SUMMARY TABLE')
print('=' * 70)
print(f'{"Method":<45} {"EM":>8} {"Valid":>12}')
print('-' * 70)
for f in sorted(glob.glob('outputs/**/*.metrics*.json', recursive=True)):
    with open(f) as fh:
        m = json.load(fh)
    name = f.replace('outputs/', '').replace('.metrics.json', '').replace('.metrics.backoff.json', ' (backoff)')
    print(f'{name:<45} {m["overall"]["em"]:>8.3f} {m["overall"]["num_valid_answer"]:>12}')